In [ ]:
pip install pyserial

In [ ]:
import serial.tools.list_ports

ports = serial.tools.list_ports.comports()
for port in ports:
    print(f"Device found: {port.device} - {port.description}")

In [ ]:
SERIAL_PORT = '/dev/cu.usbmodemSDA6C2E1E721' 
BAUD_RATE = 115200

In [ ]:
score = 0
combo = 0
max_combo = 0

DIR_MAP = {
    'R': '➡️ Right (+X)',
    'L': '⬅️ Left (-X)',
    'U': '⬆️ Up (+Y)',
    'D': '⬇️ Down (-Y)'
}

def process_hit(hit_type, hit_direction):
    global score, combo, max_combo
    
    dir_text = DIR_MAP.get(hit_direction, f"❓ Unknown({hit_direction})")
    
    if hit_type == "P":
        score += 100
        combo += 1
        print(f"[{dir_text}] 🕺 PERFECT! (+100) | 👑 Combo: {combo} | 💯 Total Score: {score}")
    elif hit_type == "G":
        score += 50
        combo += 1
        print(f"[{dir_text}] 👍 GOOD!    (+50)  | 👑 Combo: {combo} | 💯 Total Score: {score}")
    elif hit_type == "M":
        combo = 0
        print(f"[{dir_text}] ❌ MISS!    (+0)   | 💔 Combo Broken! | 💯 Total Score: {score}")
    
    if combo > max_combo:
        max_combo = combo

In [ ]:
print("Complete configuration. Attempting to connect to hardware...")

try:
    with serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=0.5) as ser:
        print(f"✅ Successfully connected to {SERIAL_PORT}")
        time.sleep(1.5) 
        
        ser.reset_input_buffer() 
        ser.reset_output_buffer()
        
        print("🚀 Sending START command...")
        ser.write(b'S') 
        print("-" * 50)
        print("🎮 Game started! Please swing the development board ...\n")

        while True:
            if ser.in_waiting > 0:
                try:
                    raw_line = ser.readline().decode('utf-8').strip()
                except UnicodeDecodeError:
                    continue 
                
                if not raw_line:
                    continue
                    

                if raw_line.startswith("HIT:"):
                    parts = raw_line.split(":")
                    
                    if len(parts) >= 3:
                        hit_result = parts[1]    # 'P', 'G', 'M'
                        hit_direction = parts[2] # 'R', 'L', 'U', 'D'
                        process_hit(hit_result, hit_direction)
                    else:
                        print(f"⚠️ Incomplete HIT data: {raw_line}")
                
                elif raw_line.startswith("DEBUG_SQ:"):
                    sq_val = raw_line.split(":")[1]
                    # print(f"   [DEBUG] Current dynamic acceleration square sum: {sq_val}")
                    
                else:
                    print(f"🔧 [Board Log] {raw_line}")
                    
except serial.SerialException as e:
    print(f"\n❌ Serial connection failed: {e}")
    print("Please check if the development board is properly connected, or if the serial port is being used by another program.")
    
except KeyboardInterrupt:
    print("\n" + "=" * 50)
    print("🛑 Game stopped (manually)")
    print(f"🏆 Final score: {score}")
    print(f"🔥 Max combo: {max_combo}")
    print("=" * 50)
    
    try:
        with serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=0.5) as ser:
            ser.write(b'X')
    except Exception:
        pass